# ⚛️ Full-XOR Chemistry | No Borrowed Bits

## Why this chemistry construction matters

The DF–THC task is not scored on approximation error alone. After the submitted
angle words are decoded, the verifier charges

$$
L_{\mathrm{chem}}=\lambda_{\mathrm{eff}}TQ,
\qquad
\lambda_{\mathrm{eff}}
=\sqrt{E_{\mathrm{gap}}(2\Lambda-E_{\mathrm{gap}})}.
$$

$T$ is the Toffoli-like work of one block-encoding step and $Q$ is its peak
live width. Both reconstruction errors must remain below 20 mHa, while the raw
normalization and SOS gap must remain above strict floors. The public Full-XOR
portfolio reaches **22.8058** overall; its chemistry member costs about
**339,981,000** at $Q=430$, $T=16{,}272$ and
$\lambda_{\mathrm{eff}}\approx48.590$.

### The factorization: fewer vectors, lower rank

The FeMoCo Hamiltonian is represented by eight groups. Each group contains 143
vectors and 60 coefficient channels, written $(R,B,C)=(8,143,60)$. Pruning the
older $B=154$ construction matters twice: it reduces the number of addresses
from 1,308 to 1,220 and lets inner state preparation use nine 16-way blocks
instead of ten. Reducing the coefficient rank from 62 to 60 lowers that stage
again, to 5,311 Toffolis.

Every rotation angle is stored as an 8-bit word. The words were quantized
jointly with simple Boolean structure: in eleven of the fifteen lookup tables,
the lowest bit of each loaded word is affine on every 16-address block; three
tables also have an affine second-lowest bit. An affine bit needs only XOR/CNOT
logic, so it avoids the nonlinear products that cost Toffolis.

### The loader: a true XOR oracle

Each `ANGLE_k` circuit implements

$$
|a\rangle|z\rangle\longmapsto
|a\rangle|z\oplus\delta_k(a)\rangle
$$

for every initial target word $z$. Earlier loaders temporarily cleared word
bits and borrowed them as scratch space. That is cheaper on the verifier's usual
zero-initialized stream, but it is not the full XOR contract for arbitrary
targets. Full-XOR instead targets each computed address product directly into
one affected word bit and fans it to the other affected bits with CNOTs. The
current fifteen circuits cost 4,171 Toffolis per pass; two passes contribute
8,342 of the total 16,272.

The discrete $(R,B,C)=(8,143,60)$ construction comes from SomeJoint's research
line. The current public base combines its factorization with Ahmed Berat
Özer's table weights, Mudit Chaturvedi's 895Q streamed DQI and heliosli's
fifteen verified chemistry circuits. It is assembled and re-verified in
*The dataset is public now. Enjoy!*. This notebook uses that exact verified
archive as its base.

### Portfolio release rule

The fresh heliosli archive is treated as a complete portfolio, not merely a
chemistry donor. Its invsqrt, DQI and chemistry members all improve or match
the components used in chemistry v8. The release starts from its verified
`22.8058` geometric mean before any optional chemistry micro-improvement.

## Our experiment: rotate the coefficient gauge

The 60 coefficient channels are not unique. For one group, let $X_c$ be the
matrix reconstructed from column $c$ of the weight tensor and let $W_c$ be its
identity offset. For any orthogonal matrix $O$,

$$
X'_c=\sum_d X_dO_{dc},\qquad W'_c=\sum_d W_dO_{dc}.
$$

Orthogonality preserves all quadratic physics:

$$
\sum_c X'_c\otimes X'_c=\sum_c X_c\otimes X_c,
\qquad
\sum_c W'_cX'_c=\sum_c W_cX_c,
\qquad
\sum_c(W'_c)^2=\sum_cW_c^2.
$$

Therefore the reconstructed Hamiltonian, both errors and the SOS gap should be
unchanged apart from floating-point noise. But the verifier's LCU cost contains
column-wise absolute values,

$$
\Lambda_2=\frac14\sum_c
\left(\lVert w_c\rVert_1+|w_{B,c}|\right)^2,
$$

which is not rotation invariant. We search pairwise Givens rotations of the 60
channels inside each of the eight groups, accepting only rotations that lower
this exact surrogate. The resulting factor is then evaluated and passed through
the complete official chemistry verifier. The archive is released only when
the complete cost falls by more than numerical noise; otherwise the verified
base is retained.


In [ ]:
import hashlib, json, math, shutil, sys, zipfile
from dataclasses import replace
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

INPUT, WORK = Path('/kaggle/input'), Path('/kaggle/working')
BASE_DIGEST = 'd823f654acb606d1a9563cd9fdeada24a7d03d58dab1413e45aa9b1a2f705ea9'
BASE_ARCHIVE = '27bff27313acde91d36e557f26d7c597751542c46f65b6af48cd08487bc06267'
SOURCE_ENSEMBLE = 22.8058
REFERENCE = {'invsqrt': 2076774.233715282, 'dqi': 24439463583672.93,
             'dfthcsos': 3355599588.3155365}
BASE_COST = {'invsqrt': 102349.2360480486, 'dqi': 412646000.0,
             'dfthcsos': 339981000.0}

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def member_digest(path):
    with zipfile.ZipFile(path) as archive:
        rows = sorted((name, hashlib.sha256(archive.read(name)).hexdigest())
                      for name in archive.namelist() if not name.endswith('/'))
    return hashlib.sha256(json.dumps(rows, separators=(',', ':')).encode()).hexdigest()

digests = {path: member_digest(path) for path in INPUT.rglob('submission.zip')}
base_zip = next((path for path, digest in digests.items() if digest == BASE_DIGEST), None)
assert base_zip and sha256(base_zip) == BASE_ARCHIVE, [(str(path), digest) for path, digest in digests.items()]

base_dir = Path('/tmp/open_crown_base')
candidate_dir = Path('/tmp/open_crown_candidate_chem')
for folder in (base_dir, candidate_dir):
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir()

with zipfile.ZipFile(base_zip) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) == 18
    assert all(not Path(name).is_absolute() and '..' not in Path(name).parts for name in names)
    assert archive.testzip() is None
    archive.extractall(base_dir)
shutil.copytree(base_dir / 'dfthcsos', candidate_dir, dirs_exist_ok=True)

competition = next(path for path in INPUT.rglob('google-verifiable-quantum-circuit-compilation') if path.is_dir())
sys.path[:0] = [str(competition / 'verifiers'), str(competition / 'kickmix')]
from dfthcsos.verify import load_inputs, evaluate_words, passes_accuracy, verify

with np.load(base_dir / 'dfthcsos/submission.npz', allow_pickle=False) as archive:
    factor = {name: np.array(archive[name]) for name in archive.files}
words, edges, brot = factor['angle_words'], factor['tree_edges'], int(factor['brot'])
base_inputs = load_inputs(factor)

def gauge_objective(weights, stored_offsets):
    column_l1 = np.sum(np.abs(weights), axis=1)
    return float(0.25 * np.sum((column_l1 + np.abs(stored_offsets)) ** 2))

def pair_objective(weight_pair, offset_pair):
    return float(0.25 * np.sum((np.sum(np.abs(weight_pair), axis=0) + np.abs(offset_pair)) ** 2))

rotated_w = np.array(factor['W'], dtype=np.float64, copy=True)
rotated_h2 = np.array(factor['H2_I'], dtype=np.float64, copy=True)
objective_before = gauge_objective(rotated_w, rotated_h2)
angle_grid = np.deg2rad(np.array([-8, -4, -2, -1, -.5, -.25, .25, .5, 1, 2, 4, 8], float))
accepted = []

for sweep in range(2):
    sweep_gain = 0.0
    for rho in range(rotated_w.shape[0]):
        for left in range(rotated_w.shape[2] - 1):
            for right in range(left + 1, rotated_w.shape[2]):
                weight_pair = rotated_w[rho, :, [left, right]].T
                offset_pair = rotated_h2[rho, [left, right]]
                old_cost = pair_objective(weight_pair, offset_pair)
                best_cost, best_angle, best_weights, best_offsets = old_cost, 0.0, None, None
                for angle in angle_grid:
                    cosine, sine = np.cos(angle), np.sin(angle)
                    rotation = np.array([[cosine, -sine], [sine, cosine]])
                    trial_weights = weight_pair @ rotation
                    trial_offsets = offset_pair @ rotation
                    trial_cost = pair_objective(trial_weights, trial_offsets)
                    if trial_cost < best_cost - 1e-12:
                        best_cost = trial_cost
                        best_angle = float(angle)
                        best_weights = trial_weights
                        best_offsets = trial_offsets
                if best_weights is not None:
                    rotated_w[rho, :, [left, right]] = best_weights.T
                    rotated_h2[rho, [left, right]] = best_offsets
                    gain = old_cost - best_cost
                    sweep_gain += gain
                    accepted.append({'sweep': sweep + 1, 'group': rho,
                                     'columns': [left, right],
                                     'angle_degrees': float(np.rad2deg(best_angle)),
                                     'objective_gain': gain})
    if sweep_gain < 1e-10:
        break

# The coarse sweep identifies promising channel pairs. Refine only those pairs
# with small incremental rotations; this is cheap and avoids a dense scan of
# all 14,160 pair locations.
fine_pairs = sorted({(row['group'], *row['columns']) for row in accepted})
fine_grid = np.deg2rad(np.arange(-0.24, 0.241, 0.01))
for rho, left, right in fine_pairs:
    weight_pair = rotated_w[rho, :, [left, right]].T
    offset_pair = rotated_h2[rho, [left, right]]
    old_cost = pair_objective(weight_pair, offset_pair)
    best_cost, best_angle, best_weights, best_offsets = old_cost, 0.0, None, None
    for angle in fine_grid:
        cosine, sine = np.cos(angle), np.sin(angle)
        rotation = np.array([[cosine, -sine], [sine, cosine]])
        trial_weights = weight_pair @ rotation
        trial_offsets = offset_pair @ rotation
        trial_cost = pair_objective(trial_weights, trial_offsets)
        if trial_cost < best_cost - 1e-14:
            best_cost = trial_cost
            best_angle = float(angle)
            best_weights = trial_weights
            best_offsets = trial_offsets
    if best_weights is not None:
        rotated_w[rho, :, [left, right]] = best_weights.T
        rotated_h2[rho, [left, right]] = best_offsets
        accepted.append({'sweep': 'fine', 'group': rho,
                         'columns': [left, right],
                         'angle_degrees': float(np.rad2deg(best_angle)),
                         'objective_gain': old_cost - best_cost})

objective_after = gauge_objective(rotated_w, rotated_h2)
gauge_factor = dict(factor)
gauge_factor['W'] = rotated_w
gauge_factor['H2_I'] = rotated_h2
gauge_inputs = load_inputs(gauge_factor)
baseline = evaluate_words(base_inputs, words, edges, brot)
BASE_COST['dfthcsos'] = float(baseline['lambda_eff']) * 16_272.0 * 430.0
gauge_metrics = evaluate_words(gauge_inputs, words, edges, brot)
gauge_changed = objective_after < objective_before - 1e-10

# The new public factor is already pairwise-gauge sparse. Probe a second,
# independent direction: ultra-fine scales for W and H2_I. The remaining
# accuracy margin is only about 0.00055 mHa, so the grid is measured in ppb.
def metrics_at(scale_w, scale_h):
    trial_inputs = replace(
        gauge_inputs,
        w=gauge_inputs.w * scale_w,
        W=gauge_inputs.W * scale_h,
    )
    metrics = evaluate_words(trial_inputs, words, edges, brot)
    return {
        'scale_w': float(scale_w),
        'scale_h': float(scale_h),
        'pass': bool(passes_accuracy(metrics)),
        'error_1': float(metrics['error_1']),
        'error_2': float(metrics['error_2']),
        'Lambda': float(metrics['Lambda']),
        'E_gap': float(metrics['E_gap']),
        'lambda_eff': float(metrics['lambda_eff']),
    }

ppb = np.array([-100, -25, 0, 25, 100], float)
scale_trials = [
    metrics_at(1 + x * 1e-9, 1 + y * 1e-9)
    for x in ppb for y in ppb
]
best_scale = min((row for row in scale_trials if row['pass']), key=lambda row: row['lambda_eff'])
center_w = (best_scale['scale_w'] - 1) * 1e9
center_h = (best_scale['scale_h'] - 1) * 1e9
if best_scale['scale_w'] != 1.0 or best_scale['scale_h'] != 1.0:
    scale_trials += [
        metrics_at(1 + x * 1e-9, 1 + y * 1e-9)
        for x in np.arange(center_w - 10, center_w + 10.01, 2.5)
        for y in np.arange(center_h - 10, center_h + 10.01, 2.5)
    ]
best_scale = min((row for row in scale_trials if row['pass']), key=lambda row: row['lambda_eff'])

selected = dict(gauge_factor)
selected['W'] = gauge_factor['W'] * best_scale['scale_w']
selected['H2_I'] = gauge_factor['H2_I'] * best_scale['scale_h']
final_inputs = load_inputs(selected)
final_metrics = evaluate_words(final_inputs, words, edges, brot)
scale_changed = best_scale['scale_w'] != 1.0 or best_scale['scale_h'] != 1.0
changed = gauge_changed or scale_changed
candidate_report, candidate_error = None, None
if changed:
    np.savez(candidate_dir / 'submission.npz', **selected)
    try:
        candidate_report = verify(candidate_dir)
    except Exception as error:
        candidate_error = f'{type(error).__name__}: {error}'

improved = bool(changed and passes_accuracy(final_metrics)
                and candidate_report and candidate_report.get('status') == 'PASS'
                and float(candidate_report['score']) < BASE_COST['dfthcsos'] - 1e-3)
destination = WORK / 'submission.zip'
if not improved:
    shutil.copyfile(base_zip, destination)
else:
    with zipfile.ZipFile(destination, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as output:
        for path in sorted(item for item in base_dir.rglob('*') if item.is_file()):
            relative = path.relative_to(base_dir).as_posix()
            source = candidate_dir / 'submission.npz' if relative == 'dfthcsos/submission.npz' else path
            output.write(source, relative)

with zipfile.ZipFile(destination) as archive:
    release_names = archive.namelist()
    assert len(release_names) == len(set(release_names)) == 18
    assert archive.testzip() is None

chemistry_cost = float(candidate_report['score']) if improved else BASE_COST['dfthcsos']
costs = {**BASE_COST, 'dfthcsos': chemistry_cost}
ratios = {task: REFERENCE[task] / costs[task] for task in REFERENCE}
ensemble = SOURCE_ENSEMBLE * (BASE_COST['dfthcsos'] / chemistry_cost) ** (1 / 3)
receipt = {'branch': 'chemistry-factorization-gauge-qrom',
           'source': 'heliosli/the-dataset-is-public-now-enjoy',
           'base_member_digest': BASE_DIGEST, 'base_archive_sha256': BASE_ARCHIVE,
           'hypothesis': 'orthogonal coefficient gauge followed by ultra-fine W/H2_I calibration on the strongest full public portfolio',
           'baseline_metrics': baseline, 'gauge_metrics': gauge_metrics,
           'best_scale_trial': best_scale, 'final_metrics': final_metrics,
           'selected_scales': {'W': best_scale['scale_w'], 'H2_I': best_scale['scale_h']},
           'gauge_objective_before': objective_before,
           'gauge_objective_after': objective_after,
           'accepted_rotations': len(accepted),
           'largest_rotations': sorted(accepted, key=lambda row: row['objective_gain'], reverse=True)[:20],
           'candidate_verifier': candidate_report, 'candidate_error': candidate_error,
           'chemistry_improved': improved,
           'portfolio_improved': improved,
           'task_ratios': ratios, 'expected_ensemble': ensemble,
           'release_archive_sha256': sha256(destination)}
(WORK / 'chemistry_branch_receipt.json').write_text(json.dumps(receipt, indent=2))

decision = 'chemistry tuning selected' if improved else 'chemistry tuning rejected; fresh 22.8058 base retained'
objective_gain = 100 * (objective_before - objective_after) / objective_before
display(Markdown(fr"""## Result

**Decision:** {decision}  
**Expected score:** {ensemble:.6f}  
**Accepted pair rotations:** {len(accepted)}  
**Selected scales:** $s_W={best_scale['scale_w']:.10f}$,
$s_H={best_scale['scale_h']:.10f}$  
**Two-body LCU surrogate:** {objective_before:.6f} → {objective_after:.6f}
({objective_gain:+.4f}%)  
**Final errors:** {1000 * final_metrics['error_1']:.6f} mHa,
{1000 * final_metrics['error_2']:.6f} mHa

The release contains 18 unique members and passes ZIP read-back. The compact
receipt records the exact source digest, tested point, verifier result and final
archive hash.
"""))
print(json.dumps(receipt, indent=2))


## Reading the experiment

The gauge and calibration searches change only the continuous coefficient
representation. The
1,220 address vectors, all fifteen Full-XOR circuits, their 8-bit words, the
430-qubit peak and the 16,272-Toffoli step remain fixed. That separation makes
the result interpretable: any accepted gain comes from a lower LCU
normalization, not from an unverified circuit rewrite.

A null result would also be informative. It would mean the current coefficient
basis is locally sparse under the tested pairwise rotations and already pinned
to the 20 mHa boundary along the tested scale directions. The next gain would
then require a structural change: a better coded-bit assignment, fewer
nonlinear address products, or a new $(R,B,C)$ factorization.

This notebook is the chemistry branch of the three-notebook rotation. Its
experiments stay separate from the general frontier and Width Lab branches.
